# Training: a SageMaker job against moto

The Feast-built training frame goes to S3 as the `train` channel; an `Estimator` runs `train.py` in the training image and writes `model.joblib` + `metrics.json` back to S3.

In [ ]:
import io

import boto3
import pandas as pd
from california_house_pricing.feature_contract import (
    FEATURE_COLUMNS,
    LABEL_COLUMN,
)
from feast import FeatureStore

s3 = boto3.client(
    "s3", endpoint_url="http://california-house-pricing-moto:5000"
)
BUCKET = "california-house-pricing"
try:
    s3.create_bucket(Bucket=BUCKET)
except s3.exceptions.BucketAlreadyOwnedByYou:
    pass

store = FeatureStore(repo_path="feature_repo")
entities = pd.read_parquet("data/processed/house_values.parquet")
training = store.get_historical_features(
    entity_df=entities,
    features=[f"block_group_profile:{c}" for c in FEATURE_COLUMNS]
    + [f"house_value_labels:{LABEL_COLUMN}"],
).to_df()

buf = io.BytesIO()
training[FEATURE_COLUMNS + [LABEL_COLUMN]].to_csv(buf, index=False)
s3.put_object(Bucket=BUCKET, Key="input/train/train.csv", Body=buf.getvalue())
len(training)

## The job

Image, role, hyperparameters — the same call shape the pipeline step reuses.

In [ ]:
import json

from sagemaker.estimator import Estimator
from sagemaker.local import LocalSession

session = LocalSession()
estimator = Estimator(
    image_uri="california-house-pricing:train",
    role="arn:aws:iam::111111111111:role/service-role/SageMakerRole",
    instance_count=1,
    instance_type="local",
    output_path=f"s3://{BUCKET}/output",
    sagemaker_session=session,
    hyperparameters={
        "n_estimators": 400,
        "learning_rate": 0.05,
        "num_leaves": 63,
    },
)
estimator.set_entry_point(
    source_dir="src/california_house_pricing", entry_point="train.py"
)
estimator.fit({"train": f"s3://{BUCKET}/input/train"})
json.loads(
    s3.get_object(
        Bucket=BUCKET,
        Key=f"output/{estimator.latest_training_job.name}/output/metrics.json",
    )["Body"].read()
)